In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, timezone
import numpy as np

In [ ]:
i_op_deg = 70
e = 1 - 5 / 3 * np.cos(np.deg2rad(i_op_deg)) ** 2
print(f"e: {e:.4f}")
sma = 16000
r_moon = 1737.4
alt = sma * (1 - e) - r_moon
print(f"altitude: {alt:.1f} km")

## Compute Moon position in Sun-Earth Fixed Rotation Frame

In [ ]:
from src.low_energy_transfer import eci_to_serot
import pylupnt as pnt

# get moon position
et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 0, 0, 0), pnt.UTC, pnt.TAI)
pnt.set_lupnt_epoch(0.0)
tspan = np.linspace(0, 3600 * 24 * 29.53, 3601)  # 30 days
et = et0 + tspan

moon_rv = pnt.get_body_pos_vel(et, pnt.EARTH, pnt.MOON, pnt.ECI)

moon_se = eci_to_serot(et, moon_rv)  # moon position in the Sun-Earth rotating frame
theta = np.arctan2(moon_se[:, 1], moon_se[:, 0])
theta[theta < 0] += 2 * np.pi  # 0 to 2pi
print("Theta (deg):", np.rad2deg(theta))

# closest to 0, 90, 180, 270 deg
idxs = [np.argmin(np.abs(np.rad2deg(theta) - angle)) for angle in [0, 90, 180, 270]]

import matplotlib.pyplot as plt

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111)
ax.plot(moon_se[:, 0], moon_se[:, 1], "k-")
ax.plot(moon_se[0, 0], moon_se[0, 1], "ro", label="Start (2030-01-01)")
ax.plot(moon_se[-1, 0], moon_se[-1, 1], "bo", label="End (2030-01-31)")
for idx in idxs:
    ax.plot(moon_se[idx, 0], moon_se[idx, 1], "go")
    ax.text(
        moon_se[idx, 0],
        moon_se[idx, 1],
        f"{np.rad2deg(theta[idx]):.1f} deg",
        fontsize=8,
    )
    print(
        f"Theta: {np.rad2deg(theta[idx]):.1f} deg,   ET: {et[idx]}   Time: {pnt.time_to_gregorian_string(et[idx], 3)}"
    )
ax.plot(0, 0, "yo", label="Earth")
ax.set_xlabel("X [km]")
ax.set_ylabel("Y [km]")
ax.set_title("Moon orbit in the Sun-Earth rotating frame (30 days)")
ax.axis("equal")
ax.grid(True)
ax.legend(loc="upper right")
plt.show()

## Check Integration Termination and Backward Propagation

In [ ]:
# test termination criteria
iparam = pnt.IntegratorParams(max_iter=1000, abstol=1e-10, reltol=1e-10)


def terminate_func(t, x):
    return np.linalg.norm(x[0:3]) < (
        pnt.R_MOON + 1000e3
    )  # terminate if altitude < 800 km


iparam.set_terminate_if(terminate_func)  # terminate if altitude < 500 km

dynamics = pnt.NBodyDynamics()
dynamics.set_integrator(pnt.IntegratorType.RKF45)
dynamics.set_integrator_params(iparam)
dynamics.add_body(pnt.Body.Moon(20, 20))
dynamics.add_body(pnt.Body.Earth())
dynamics.add_body(pnt.Body.Sun())
dynamics.set_time_step(60)
dynamics.set_frame(pnt.MOON_CI)
dynamics.set_print_progress(False)

coe = np.array(
    [pnt.R_MOON + 1000e3, 0.2, np.deg2rad(90), 0, 0, np.deg2rad(190)]
)  # 1,000 km altitude circular polar orbit at apole
rv0 = pnt.classical_to_cart(coe, pnt.GM_MOON)
print("Initial state (ECI):", rv0)

tspan = np.linspace(0, 3600 * 2, 500)  # 10 days
dynamics.propagate(rv0, et0 + tspan)
xprop_fw, info = dynamics.propagate_with_info(
    rv0, et0, et0 + tspan
)  # propagate backward for 10 days
xprop_bw, info_bw = dynamics.propagate_with_info(
    rv0, et0, et0 - tspan
)  # propagate backward for 10 days

print("Final state (ECI):", xprop_fw[-1, :])
print("Final Distance from Moon[km]:", np.linalg.norm(xprop_fw[-1, 0:3] - pnt.R_MOON))
nx_fw = xprop_fw.shape[0]
nx_bw = xprop_bw.shape[0]
print("Number of steps:", nx_fw)
print("Termination Info: ", info.terminated)
print("Termination Reason: ", info.reason)
print(
    "Is terminated by user condition?: ",
    info.reason == pnt.TerminationReason.UserCondition,
)

fig = plt.figure(figsize=(6, 4))
ax = fig.add_subplot(111)
plt.plot(
    tspan[:nx_fw] / 3600,
    (np.linalg.norm(xprop_fw[:, 0:3], axis=1) - pnt.R_MOON) / 1000,
    "ko-",
    label="Forward",
)
plt.plot(
    tspan[:nx_bw] / 3600,
    (np.linalg.norm(xprop_bw[:, 0:3], axis=1) - pnt.R_MOON) / 1000,
    "ro-",
    label="Backward",
)
plt.xlabel("Time [hours]")
plt.ylabel("Altitude [km]")
plt.title("Altitude vs Time (terminated if altitude smaller than 1000 km)")
plt.grid(True)
plt.legend()
plt.show()

## Test Low-Energy Transfer

In [ ]:
import pylupnt as pnt
import numpy as np
import os

# get moon position
et0 = 948341083.72  # where the moon is at 0 deg in the Sun-Earth rotating frame

n_sma = 7  # number of semi-major axis samples 4000 km - 16000 km (4000, 6000, 8000, 10000, 12000, 14000, 16000)
n_incs = 6  # number of inclination samples (39.2, 40, 45, 50, 55, 60)
n_Omega = 1  # number of RAAN samples
n_et = 16  # number of epoch samples

n_sat = n_sma * n_incs * n_Omega
coes = np.zeros((n_sat, 6))

smas = np.linspace(4000, 16000, n_sma) * 1e3  # km -> m
incs = np.deg2rad(np.array([40, 45, 50, 55, 60, 65]))
Omegas = np.linspace(0, 2 * np.pi, n_Omega, endpoint=False)  # rad

for inc in incs:
    ecc2 = 1 - 5 / 3 * np.cos(inc) ** 2
    print(f"inc: {np.rad2deg(inc)} deg,   ecc: {np.sqrt(ecc2)}")

idx = 0
for i, sma in enumerate(smas):
    for j, inc in enumerate(incs):
        for k, Omega in enumerate(Omegas):
            ecc2 = 1 - 5 / 3 * np.cos(inc) ** 2
            if ecc2 < 0:
                continue  # skip invalid ecc

            coes[idx, 0] = sma
            coes[idx, 1] = np.sqrt(ecc2)
            coes[idx, 2] = inc
            coes[idx, 3] = Omega
            coes[idx, 4] = np.deg2rad(90)
            coes[idx, 5] = 0.0

            idx += 1

tspan = np.linspace(0, 3600 * 24 * 29.53, n_et, endpoint=False)  # 30 days
et0s = et0 + tspan

min_dv = 50  # m/s
max_dv = 1000  # m/s
d_dv = 1.0  # m/s
n_dv = int((max_dv - min_dv) / d_dv) + 1
dvvec = np.linspace(min_dv, max_dv, n_dv)  # m/s

# data directory
datadir = "data/lowenergy_transfer/n_sma_{}_n_inc_{}_n_Omega_{}_dv_{}_{}_{:.1f}".format(
    n_sma, n_incs, n_Omega, int(min_dv), int(max_dv), d_dv
)
os.makedirs(datadir, exist_ok=True)

# save all the parameters
np.savez(
    os.path.join(datadir, "parameters.npz"),
    coes=coes,
    et0s=et0s,
    dvvec=dvvec,
)

In [ ]:
from src.low_energy_transfer import search_low_energy_transfer

for et in et0s:
    print(
        f"Searching for low-energy transfers for et0 = {et} ({pnt.time_to_gregorian_string(et, 3)}) ..."
    )
    outdir = datadir + f"/et_{et}"
    os.makedirs(outdir, exist_ok=True)

    search_low_energy_transfer(
        [et],
        coes,
        dvvec,
        out_dir=outdir,
        prop_days=160,
        max_alt=1e4 * 1e3,
        debug=True,
        debug_pc=False,
        n_jobs=10,
    )